# Phase 1 Dataset Metrics Screening Review

This workbook loads the aggregate phase-1 screening output for the 100 sampled loci, plots the main dataset metrics, and selects a representative subset of genes spread across the joint metric distribution.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    candidates = []
    current = (start or Path.cwd()).resolve()
    candidates.extend([current, *current.parents])

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (
            (candidate / 'utils').is_dir()
            and (candidate / 'vignettes').is_dir()
            and (candidate / 'README.md').exists()
        ):
            return candidate

    raise FileNotFoundError('Could not locate eQTL_annotations_for_susine project root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

MANIFEST_PATH = PROJECT_ROOT / 'config' / 'loci_manifest_sample_100_per_chrom.csv'
CSV_PATH = PROJECT_ROOT / 'output' / 'prelim' / 'loci_manifest_sample_100_per_chrom_phase1_dataset_metrics.csv'
PLOT_DIR = PROJECT_ROOT / 'output' / 'prelim' / 'phase1_metrics_screening_review'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

METRIC_COLUMNS = ['M1', 'M2', 'M4', 'z_count_abs_gt_3', 'z_eff_signals']
TARGET_SAMPLE_SIZE = 10
MIN_PER_CHROM = 2
SAMPLING_METHOD = 'k_medoids'  # 'k_medoids', 'maximin', or 'hybrid'
HYBRID_SPREAD_COUNT = 4
HYBRID_REPRESENTATIVE_COUNT = 8
DISTANCE_METRIC = 'mahalanobis'  # 'euclidean' or 'mahalanobis' for the sampling distance
DISTANCE_LOSS = 'squared'  # 'squared' or 'plain' for the sampler objective

manifest_df = pd.read_csv(MANIFEST_PATH)
metrics_df = pd.read_csv(CSV_PATH)
metrics_df = metrics_df[metrics_df['status'].isin(['completed', 'skipped_existing'])].copy()
metrics_df = metrics_df.sort_values(['gene_name', 'locus_id']).reset_index(drop=True)

missing_metrics = [col for col in METRIC_COLUMNS if col not in metrics_df.columns]
if missing_metrics:
    raise ValueError(f'Missing metric columns: {missing_metrics}')
if metrics_df[METRIC_COLUMNS].isna().any().any():
    raise ValueError('Metric columns contain missing values; resolve before plotting or sampling.')

print(f'Loaded source manifest: {MANIFEST_PATH}')
print(f'Loaded {len(metrics_df)} loci from {CSV_PATH}')
print(f'Plots and sampled-gene tables will be written to: {PLOT_DIR}')

In [ ]:
summary_df = metrics_df[['locus_id', 'gene_name', 'gene_id', 'gtex_tissue', 'gtex_chrom', *METRIC_COLUMNS]].copy()
display(summary_df.head(10))
display(metrics_df[METRIC_COLUMNS].describe().T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, metric in zip(axes, METRIC_COLUMNS):
    ax.hist(metrics_df[metric], bins=20, color='#4C78A8', edgecolor='white', linewidth=0.8)
    ax.axvline(metrics_df[metric].median(), color='#E45756', linestyle='--', linewidth=1.5, label='median')
    ax.set_title(metric)
    ax.set_xlabel(metric)
    ax.set_ylabel('Count')
    ax.legend(frameon=False)

fig.suptitle('Phase 1 Dataset Metric Histograms', fontsize=14)
fig.tight_layout()
hist_path = PLOT_DIR / 'phase1_metric_histograms.png'
fig.savefig(hist_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Wrote: {hist_path}')

In [ ]:
n_metrics = len(METRIC_COLUMNS)
fig, axes = plt.subplots(n_metrics, n_metrics, figsize=(14, 14))

for i, y_metric in enumerate(METRIC_COLUMNS):
    for j, x_metric in enumerate(METRIC_COLUMNS):
        ax = axes[i, j]
        if i == j:
            ax.hist(metrics_df[x_metric], bins=20, color='#72B7B2', edgecolor='white', linewidth=0.7)
        else:
            ax.scatter(metrics_df[x_metric], metrics_df[y_metric], s=22, alpha=0.75, color='#4C78A8')
        if i == n_metrics - 1:
            ax.set_xlabel(x_metric)
        else:
            ax.set_xticklabels([])
        if j == 0:
            ax.set_ylabel(y_metric)
        else:
            ax.set_yticklabels([])

fig.suptitle('Pairwise Scatter Plot Matrix for Phase 1 Dataset Metrics', fontsize=14)
fig.tight_layout()
scatter_path = PLOT_DIR / 'phase1_metric_scatter_matrix.png'
fig.savefig(scatter_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Wrote: {scatter_path}')

## Representative Sampling Strategy

The goal is to sample roughly 8-16 genes that cover the joint distribution of the metrics rather than only selecting extreme outliers.

Strategy used here:
1. Keep `M1`, `M2`, and `M4` on their native scale.
2. Apply `log1p` to `z_count_abs_gt_3` and `z_eff_signals` because those are right-skewed.
3. Robust-scale each metric using the median and IQR.
4. Run either a global `k`-medoids sampler, a global maximin sampler, or a hybrid of the two in the 5D transformed space, using either Euclidean or Mahalanobis distance.
5. For the `k`-medoids path, repair the selected set with chromosome-constrained swaps so each chromosome contributes at least `MIN_PER_CHROM` loci.
6. For the hybrid path, pick a representative core with constrained `k`-medoids and then add spread-seeking loci with constrained maximin.
7. Use the resulting selected loci as representative genes for distinct regions of metric space.

This gives a representative spread over the joint metric distribution while keeping each selected row tied to a real gene/locus. Mahalanobis distance discounts directions where metrics co-vary strongly, while Euclidean distance treats each feature axis independently after scaling.

In [ ]:
def transform_metric_space(df: pd.DataFrame) -> pd.DataFrame:
    transformed = pd.DataFrame(index=df.index)
    transformed['M1'] = df['M1']
    transformed['M2'] = df['M2']
    transformed['M4'] = df['M4']
    transformed['log1p_z_count_abs_gt_3'] = np.log1p(df['z_count_abs_gt_3'])
    transformed['z_eff_signals'] = df['z_eff_signals']
    return transformed


def robust_scale(df: pd.DataFrame) -> pd.DataFrame:
    scaled = pd.DataFrame(index=df.index)
    for col in df.columns:
        median = df[col].median()
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        scale = iqr if iqr > 0 else df[col].std(ddof=0)
        if not np.isfinite(scale) or scale == 0:
            scale = 1.0
        scaled[col] = (df[col] - median) / scale
    return scaled


def pairwise_distances(
    X: np.ndarray,
    Y: np.ndarray | None = None,
    *,
    metric: str = 'euclidean',
    VI: np.ndarray | None = None,
) -> np.ndarray:
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    if metric == 'euclidean':
        dist_sq = np.einsum('ijk,ijk->ij', diff, diff)
    elif metric == 'mahalanobis':
        if VI is None:
            raise ValueError('VI must be provided for Mahalanobis distance.')
        dist_sq = np.einsum('ijk,kl,ijl->ij', diff, VI, diff)
    else:
        raise ValueError(f'Unsupported distance metric: {metric}')
    dist_sq = np.maximum(dist_sq, 0.0)
    if DISTANCE_LOSS == 'plain':
        return np.sqrt(dist_sq)
    if DISTANCE_LOSS == 'squared':
        return dist_sq
    raise ValueError(f'Unsupported DISTANCE_LOSS: {DISTANCE_LOSS}')


def initialize_medoids(
    X: np.ndarray,
    k: int,
    *,
    metric: str = 'euclidean',
    VI: np.ndarray | None = None,
) -> list[int]:
    center = X.mean(axis=0)
    if metric == 'euclidean':
        center_dist_sq = ((X - center) ** 2).sum(axis=1)
    elif metric == 'mahalanobis':
        if VI is None:
            raise ValueError('VI must be provided for Mahalanobis distance.')
        center_diff = X - center
        center_dist_sq = np.einsum('ij,jk,ik->i', center_diff, VI, center_diff)
    else:
        raise ValueError(f'Unsupported distance metric: {metric}')
    medoids = [int(np.argmin(center_dist_sq))]
    while len(medoids) < k:
        distances = pairwise_distances(X, X[medoids], metric=metric, VI=VI)
        min_dist = distances.min(axis=1)
        min_dist[medoids] = -np.inf
        medoids.append(int(np.argmax(min_dist)))
    return medoids


def assign_clusters(
    X: np.ndarray,
    medoids: list[int],
    *,
    metric: str = 'euclidean',
    VI: np.ndarray | None = None,
) -> np.ndarray:
    return pairwise_distances(X, X[medoids], metric=metric, VI=VI).argmin(axis=1)


def update_medoids(
    X: np.ndarray,
    labels: np.ndarray,
    medoids: list[int],
    *,
    metric: str = 'euclidean',
    VI: np.ndarray | None = None,
) -> list[int]:
    new_medoids = []
    for cluster_id in range(len(medoids)):
        members = np.where(labels == cluster_id)[0]
        if len(members) == 0:
            new_medoids.append(medoids[cluster_id])
            continue
        within = pairwise_distances(X[members], metric=metric, VI=VI)
        medoid_local = int(np.argmin(within.sum(axis=1)))
        new_medoids.append(int(members[medoid_local]))
    return new_medoids


def k_medoids_sample(
    X: np.ndarray,
    k: int,
    *,
    metric: str = 'euclidean',
    VI: np.ndarray | None = None,
    max_iter: int = 25,
) -> tuple[list[int], np.ndarray]:
    medoids = initialize_medoids(X, k, metric=metric, VI=VI)
    for _ in range(max_iter):
        labels = assign_clusters(X, medoids, metric=metric, VI=VI)
        new_medoids = update_medoids(X, labels, medoids, metric=metric, VI=VI)
        if new_medoids == medoids:
            break
        medoids = new_medoids
    labels = assign_clusters(X, medoids, metric=metric, VI=VI)
    return medoids, labels


def total_assignment_cost(
    X: np.ndarray,
    medoids: list[int],
    *,
    metric: str = 'euclidean',
    VI: np.ndarray | None = None,
) -> float:
    distances = pairwise_distances(X, X[medoids], metric=metric, VI=VI)
    return float(distances.min(axis=1).sum())


def chromosome_counts(indices: list[int], chrom_series: pd.Series) -> dict[str, int]:
    counts = chrom_series.iloc[indices].value_counts().to_dict()
    return {chrom: int(counts.get(chrom, 0)) for chrom in sorted(chrom_series.unique())}


def repair_medoids_with_chrom_minimum(
    X: np.ndarray,
    medoids: list[int],
    chrom_series: pd.Series,
    *,
    min_per_chrom: int,
    metric: str = 'euclidean',
    VI: np.ndarray | None = None,
    locked_indices: list[int] | None = None,
    max_passes: int = 100,
) -> list[int]:
    if min_per_chrom < 0:
        raise ValueError('min_per_chrom must be >= 0')
    chrom_values = sorted(chrom_series.unique())
    if min_per_chrom * len(chrom_values) > len(medoids):
        raise ValueError('Impossible chromosome minimum for the requested sample size.')

    medoids = list(medoids)
    locked_set = set() if locked_indices is None else {int(idx) for idx in locked_indices}
    for _ in range(max_passes):
        counts = chromosome_counts(medoids, chrom_series)
        underrepresented = [chrom for chrom in chrom_values if counts[chrom] < min_per_chrom]
        if not underrepresented:
            break

        swap_made = False
        current_cost = total_assignment_cost(X, medoids, metric=metric, VI=VI)
        medoid_set = set(medoids)

        for target_chrom in underrepresented:
            candidate_indices = [
                idx for idx, chrom in chrom_series.items()
                if chrom == target_chrom and idx not in medoid_set
            ]
            removable_indices = [
                idx for idx in medoids
                if idx not in locked_set
                and chromosome_counts([m for m in medoids if m != idx], chrom_series)[chrom_series.iloc[idx]] >= min_per_chrom
            ]
            if not removable_indices:
                removable_indices = [
                    idx for idx in medoids
                    if chromosome_counts([m for m in medoids if m != idx], chrom_series)[chrom_series.iloc[idx]] >= min_per_chrom
                ]
            best_swap = None
            best_cost = None

            for candidate_idx in candidate_indices:
                for remove_idx in removable_indices:
                    if chrom_series.iloc[remove_idx] == target_chrom:
                        continue
                    trial_medoids = [candidate_idx if m == remove_idx else m for m in medoids]
                    trial_cost = total_assignment_cost(X, trial_medoids, metric=metric, VI=VI)
                    if best_cost is None or trial_cost < best_cost:
                        best_cost = trial_cost
                        best_swap = (remove_idx, candidate_idx)

            if best_swap is not None:
                remove_idx, candidate_idx = best_swap
                medoids = [candidate_idx if m == remove_idx else m for m in medoids]
                swap_made = True
                break

        if not swap_made:
            raise RuntimeError('Could not satisfy chromosome minimum with valid swaps.')
    else:
        raise RuntimeError('Exceeded max_passes while repairing chromosome minimums.')

    improved = True
    while improved:
        improved = False
        current_counts = chromosome_counts(medoids, chrom_series)
        current_cost = total_assignment_cost(X, medoids, metric=metric, VI=VI)
        medoid_set = set(medoids)
        best_swap = None
        best_cost = current_cost

        prioritized_remove = [idx for idx in medoids if idx not in locked_set] + [idx for idx in medoids if idx in locked_set]
        for remove_idx in prioritized_remove:
            remove_chrom = chrom_series.iloc[remove_idx]
            if current_counts[remove_chrom] <= min_per_chrom:
                continue
            for candidate_idx in range(len(chrom_series)):
                if candidate_idx in medoid_set:
                    continue
                candidate_chrom = chrom_series.iloc[candidate_idx]
                trial_counts = current_counts.copy()
                trial_counts[remove_chrom] -= 1
                trial_counts[candidate_chrom] += 1
                if any(count < min_per_chrom for count in trial_counts.values()):
                    continue
                trial_medoids = [candidate_idx if m == remove_idx else m for m in medoids]
                trial_cost = total_assignment_cost(X, trial_medoids, metric=metric, VI=VI)
                if trial_cost + 1e-12 < best_cost:
                    best_cost = trial_cost
                    best_swap = (remove_idx, candidate_idx)

        if best_swap is not None:
            remove_idx, candidate_idx = best_swap
            medoids = [candidate_idx if m == remove_idx else m for m in medoids]
            improved = True

    return sorted(medoids)


def constrained_maximin_sample(
    X: np.ndarray,
    chrom_series: pd.Series,
    *,
    k: int,
    min_per_chrom: int,
    metric: str = 'euclidean',
    VI: np.ndarray | None = None,
    seed_indices: list[int] | None = None,
    eligible_indices: list[int] | None = None,
) -> list[int]:
    chrom_values = sorted(chrom_series.unique())
    if min_per_chrom * len(chrom_values) > k:
        raise ValueError('Impossible chromosome minimum for the requested sample size.')

    center = X.mean(axis=0)
    if metric == 'euclidean':
        center_score = ((X - center) ** 2).sum(axis=1)
    elif metric == 'mahalanobis':
        if VI is None:
            raise ValueError('VI must be provided for Mahalanobis distance.')
        center_diff = X - center
        center_score = np.einsum('ij,jk,ik->i', center_diff, VI, center_diff)
    else:
        raise ValueError(f'Unsupported distance metric: {metric}')

    eligible_set = set(range(len(chrom_series))) if eligible_indices is None else {int(idx) for idx in eligible_indices}
    if seed_indices is None:
        eligible_center = [(idx, center_score[idx]) for idx in eligible_set]
        selected = [int(min(eligible_center, key=lambda item: item[1])[0])]
    else:
        selected = sorted({int(idx) for idx in seed_indices})
        if len(selected) > k:
            raise ValueError('seed_indices cannot exceed requested sample size.')
    while len(selected) < k:
        counts = chromosome_counts(selected, chrom_series)
        slots_remaining = k - len(selected)
        needed_total = sum(max(0, min_per_chrom - counts[chrom]) for chrom in chrom_values)
        candidate_pool = []
        for idx in sorted(eligible_set):
            if idx in selected:
                continue
            candidate_chrom = chrom_series.iloc[idx]
            trial_counts = counts.copy()
            trial_counts[candidate_chrom] += 1
            remaining_after_pick = slots_remaining - 1
            needed_after_pick = sum(max(0, min_per_chrom - trial_counts[chrom]) for chrom in chrom_values)
            if needed_after_pick <= remaining_after_pick:
                candidate_pool.append(idx)

        if not candidate_pool:
            raise RuntimeError('No feasible candidates remain for constrained maximin selection.')

        distances = pairwise_distances(X[candidate_pool], X[selected], metric=metric, VI=VI)
        min_dist = distances.min(axis=1)
        best_pos = int(np.argmax(min_dist))
        selected.append(int(candidate_pool[best_pos]))

    return sorted(selected)


transformed_df = transform_metric_space(metrics_df)
scaled_df = robust_scale(transformed_df)
X = scaled_df.to_numpy(dtype=float)

if DISTANCE_METRIC not in {'euclidean', 'mahalanobis'}:
    raise ValueError("DISTANCE_METRIC must be 'euclidean' or 'mahalanobis'.")
if DISTANCE_LOSS not in {'plain', 'squared'}:
    raise ValueError("DISTANCE_LOSS must be 'plain' or 'squared'.")
if SAMPLING_METHOD not in {'k_medoids', 'maximin', 'hybrid'}:
    raise ValueError("SAMPLING_METHOD must be 'k_medoids', 'maximin', or 'hybrid'.")
VI = None
if DISTANCE_METRIC == 'mahalanobis':
    covariance = np.cov(X, rowvar=False)
    VI = np.linalg.pinv(covariance)

if TARGET_SAMPLE_SIZE < 1 or TARGET_SAMPLE_SIZE > len(metrics_df):
    raise ValueError('TARGET_SAMPLE_SIZE must be between 1 and the number of loci.')
if MIN_PER_CHROM * metrics_df['gtex_chrom'].nunique() > TARGET_SAMPLE_SIZE:
    raise ValueError('MIN_PER_CHROM is too large for the requested sample size.')
if HYBRID_SPREAD_COUNT < 0 or HYBRID_SPREAD_COUNT > TARGET_SAMPLE_SIZE:
    raise ValueError('HYBRID_SPREAD_COUNT must be between 0 and TARGET_SAMPLE_SIZE.')
if HYBRID_REPRESENTATIVE_COUNT < 0 or HYBRID_REPRESENTATIVE_COUNT > TARGET_SAMPLE_SIZE:
    raise ValueError('HYBRID_REPRESENTATIVE_COUNT must be between 0 and TARGET_SAMPLE_SIZE.')
if SAMPLING_METHOD == 'hybrid' and HYBRID_SPREAD_COUNT + HYBRID_REPRESENTATIVE_COUNT != TARGET_SAMPLE_SIZE:
    raise ValueError('HYBRID_SPREAD_COUNT + HYBRID_REPRESENTATIVE_COUNT must equal TARGET_SAMPLE_SIZE.')

if SAMPLING_METHOD == 'k_medoids':
    initial_medoid_idx, initial_cluster_labels = k_medoids_sample(
        X,
        TARGET_SAMPLE_SIZE,
        metric=DISTANCE_METRIC,
        VI=VI,
    )
    selected_idx = repair_medoids_with_chrom_minimum(
        X,
        initial_medoid_idx,
        metrics_df['gtex_chrom'],
        min_per_chrom=MIN_PER_CHROM,
        metric=DISTANCE_METRIC,
        VI=VI,
    )
elif SAMPLING_METHOD == 'maximin':
    selected_idx = constrained_maximin_sample(
        X,
        metrics_df['gtex_chrom'],
        k=TARGET_SAMPLE_SIZE,
        min_per_chrom=MIN_PER_CHROM,
        metric=DISTANCE_METRIC,
        VI=VI,
    )
else:
    spread_idx = constrained_maximin_sample(
        X,
        metrics_df['gtex_chrom'],
        k=HYBRID_SPREAD_COUNT,
        min_per_chrom=0,
        metric=DISTANCE_METRIC,
        VI=VI,
    )
    remaining_idx = [idx for idx in range(len(metrics_df)) if idx not in set(spread_idx)]
    representative_local_idx, _ = k_medoids_sample(
        X[remaining_idx],
        HYBRID_REPRESENTATIVE_COUNT,
        metric=DISTANCE_METRIC,
        VI=VI,
    )
    representative_idx = sorted([remaining_idx[idx] for idx in representative_local_idx])
    selected_idx = repair_medoids_with_chrom_minimum(
        X,
        spread_idx + representative_idx,
        metrics_df['gtex_chrom'],
        min_per_chrom=MIN_PER_CHROM,
        metric=DISTANCE_METRIC,
        VI=VI,
        locked_indices=spread_idx,
    )
cluster_labels = assign_clusters(X, selected_idx, metric=DISTANCE_METRIC, VI=VI)

sampled_df = metrics_df.iloc[sorted(selected_idx)].copy()
sampled_df['sampling_method'] = f'{SAMPLING_METHOD}_on_robust_scaled_5d_metric_space_{DISTANCE_METRIC}_chrom_min_{MIN_PER_CHROM}'
sampled_df['target_sample_size'] = TARGET_SAMPLE_SIZE
sampled_df['hybrid_spread_count'] = HYBRID_SPREAD_COUNT if SAMPLING_METHOD == 'hybrid' else np.nan
sampled_df['hybrid_representative_count'] = HYBRID_REPRESENTATIVE_COUNT if SAMPLING_METHOD == 'hybrid' else np.nan
selected_chrom_counts = sampled_df['gtex_chrom'].value_counts().sort_index()

cluster_df = metrics_df.copy()
cluster_df['cluster_id'] = cluster_labels
sampled_df['cluster_id'] = cluster_labels[sampled_df.index]
representative_gene_map = sampled_df.set_index('cluster_id')['gene_name'].to_dict()

cluster_summary = (
    cluster_df.groupby('cluster_id')
    .agg(
        n_loci=('locus_id', 'size'),
        M1_min=('M1', 'min'),
        M1_max=('M1', 'max'),
        M2_min=('M2', 'min'),
        M2_max=('M2', 'max'),
        M4_min=('M4', 'min'),
        M4_max=('M4', 'max'),
        z_count_min=('z_count_abs_gt_3', 'min'),
        z_count_max=('z_count_abs_gt_3', 'max'),
        z_eff_min=('z_eff_signals', 'min'),
        z_eff_max=('z_eff_signals', 'max'),
    )
    .reset_index()
    .assign(representative_gene=lambda df: df['cluster_id'].map(representative_gene_map))
    .sort_values('cluster_id')
)

sample_path = PLOT_DIR / f'representative_gene_sample_n{TARGET_SAMPLE_SIZE}.csv'
cluster_path = PLOT_DIR / f'representative_gene_sample_n{TARGET_SAMPLE_SIZE}_cluster_summary.csv'
annotation_selection_path = PLOT_DIR / f'representative_gene_sample_n{TARGET_SAMPLE_SIZE}_annotation_selection.csv'
selected_manifest_path = PLOT_DIR / f'representative_gene_sample_n{TARGET_SAMPLE_SIZE}_manifest.csv'
sampled_df.to_csv(sample_path, index=False)
cluster_summary.to_csv(cluster_path, index=False)
selected_manifest_df = manifest_df[manifest_df['locus_id'].isin(sampled_df['locus_id'])].copy()
selected_manifest_df = selected_manifest_df.merge(
    sampled_df[['locus_id']],
    on='locus_id',
    how='inner',
)
if len(selected_manifest_df) != len(sampled_df):
    missing_manifest_loci = sorted(set(sampled_df['locus_id']) - set(selected_manifest_df['locus_id']))
    raise ValueError(f'Selected loci missing from MANIFEST_PATH: {missing_manifest_loci}')
selected_manifest_df.to_csv(selected_manifest_path, index=False)
annotation_selection_df = sampled_df[
    ['locus_id', 'gene_name', 'gene_id', 'gtex_tissue', 'gtex_chrom']
].copy()
annotation_selection_df = annotation_selection_df.assign(
    annotate=True,
    notes=(
        'selected_from_phase1_metrics:'
        + sampled_df['sampling_method'].astype(str)
    ),
    priority=np.arange(1, len(annotation_selection_df) + 1, dtype=int),
    annotation_gene_name_override=pd.NA,
    annotation_gene_id_override=pd.NA,
    annotation_tissue_override=pd.NA,
)[[
    'locus_id',
    'annotate',
    'notes',
    'priority',
    'annotation_gene_name_override',
    'annotation_gene_id_override',
    'annotation_tissue_override',
]]
annotation_selection_df.to_csv(annotation_selection_path, index=False)

display(sampled_df[['locus_id', 'gene_name', 'gene_id', 'gtex_tissue', 'gtex_chrom', *METRIC_COLUMNS]])
display(cluster_summary)
print('Selected chromosome counts:')
for chrom, count in selected_chrom_counts.items():
    print(f"- {chrom}: {count}")
print('Selected representative genes:')
for _, row in sampled_df.sort_values(['gtex_chrom', 'gene_name']).iterrows():
    print(f"- {row['gene_name']} ({row['gtex_chrom']})")
print(f'Wrote: {sample_path}')
print(f'Wrote: {cluster_path}')
print(f'Wrote: {annotation_selection_path}')
print(f'Wrote: {selected_manifest_path}')

In [ ]:
X_centered = X - X.mean(axis=0, keepdims=True)
U, S, VT = np.linalg.svd(X_centered, full_matrices=False)
max_pcs = min(5, VT.shape[0])
pca_scores = X_centered @ VT.T[:, :max_pcs]
explained_variance = (S ** 2) / max(len(X) - 1, 1)
explained_variance_ratio = explained_variance / explained_variance.sum()
variance_df = pd.DataFrame(
    {
        'PC': [f'PC{i}' for i in range(1, max_pcs + 1)],
        'explained_variance_ratio': explained_variance_ratio[:max_pcs],
        'explained_variance_percent': 100.0 * explained_variance_ratio[:max_pcs],
    }
)
display(variance_df)

loading_feature_names = list(scaled_df.columns)
pc_loading_count = min(2, max_pcs)
loading_df = pd.DataFrame(
    VT.T[:, :pc_loading_count],
    index=loading_feature_names,
    columns=[f'PC{i}' for i in range(1, pc_loading_count + 1)],
)
loading_df['abs_PC1'] = loading_df['PC1'].abs()
if pc_loading_count >= 2:
    loading_df['abs_PC2'] = loading_df['PC2'].abs()
display(loading_df)

plot_df = metrics_df.copy()
plot_df['PC1'] = pca_scores[:, 0]
plot_df['PC2'] = pca_scores[:, 1]
if max_pcs >= 4:
    plot_df['PC3'] = pca_scores[:, 2]
    plot_df['PC4'] = pca_scores[:, 3]
plot_df['selected'] = False
plot_df.loc[sampled_df.index, 'selected'] = True

chrom_values = sorted(plot_df['gtex_chrom'].unique())
chrom_cmap = plt.get_cmap('tab10')
chrom_color_map = {chrom: chrom_cmap(i % 10) for i, chrom in enumerate(chrom_values)}

fig, ax = plt.subplots(figsize=(9, 7))
for chrom in chrom_values:
    chrom_df = plot_df[plot_df['gtex_chrom'] == chrom]
    ax.scatter(
        chrom_df['PC1'],
        chrom_df['PC2'],
        s=34,
        color=chrom_color_map[chrom],
        alpha=0.75,
        label=chrom,
    )

selected_df = plot_df[plot_df['selected']].copy()
ax.scatter(
    selected_df['PC1'],
    selected_df['PC2'],
    s=90,
    facecolors='none',
    edgecolors='black',
    linewidth=1.0,
    label='Selected representatives',
)

for _, row in selected_df.iterrows():
    ax.text(row['PC1'] + 0.03, row['PC2'] + 0.03, row['gene_name'], fontsize=8)

ax.set_title(f'Representative Sample in Joint Metric Space (n={TARGET_SAMPLE_SIZE})')
ax.set_xlabel(f'PC1 ({variance_df.loc[0, "explained_variance_percent"]:.1f}% variance explained)')
ax.set_ylabel(f'PC2 ({variance_df.loc[1, "explained_variance_percent"]:.1f}% variance explained)')
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
selection_plot_path = PLOT_DIR / f'representative_gene_sample_n{TARGET_SAMPLE_SIZE}_pca.png'
fig.savefig(selection_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Wrote: {selection_plot_path}')

if max_pcs >= 4:
    fig, ax = plt.subplots(figsize=(9, 7))
    for chrom in chrom_values:
        chrom_df = plot_df[plot_df['gtex_chrom'] == chrom]
        ax.scatter(
            chrom_df['PC3'],
            chrom_df['PC4'],
            s=34,
            color=chrom_color_map[chrom],
            alpha=0.75,
            label=chrom,
        )

    ax.scatter(
        selected_df['PC3'],
        selected_df['PC4'],
        s=90,
        facecolors='none',
        edgecolors='black',
        linewidth=1.0,
        label='Selected representatives',
    )

    for _, row in selected_df.iterrows():
        ax.text(row['PC3'] + 0.03, row['PC4'] + 0.03, row['gene_name'], fontsize=8)

    ax.set_title(f'Representative Sample in Joint Metric Space (PC3 vs PC4, n={TARGET_SAMPLE_SIZE})')
    ax.set_xlabel(f'PC3 ({variance_df.loc[2, "explained_variance_percent"]:.1f}% variance explained)')
    ax.set_ylabel(f'PC4 ({variance_df.loc[3, "explained_variance_percent"]:.1f}% variance explained)')
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    selection_plot_path_pc34 = PLOT_DIR / f'representative_gene_sample_n{TARGET_SAMPLE_SIZE}_pca_pc3_pc4.png'
    fig.savefig(selection_plot_path_pc34, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Wrote: {selection_plot_path_pc34}')